In [25]:
import pandas as pd 
import matplotlib.pyplot as plt 
import utils
import torch 
from torch.utils.data import TensorDataset, DataLoader
import random
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch.nn as nn 

In [19]:
global_dataset = pd.read_csv('./datasets/global_test_dataset.csv')
global_dataset.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,-0.135189,-0.353808,-0.010601,-0.008254,-0.077122,-0.006218,-0.201272,-0.216044,-0.219519,-0.172305,...,0.003110,-0.144686,-0.084676,-0.150402,-0.131186,-0.299561,-0.156966,-0.300753,-0.265660,1
1,-0.416792,1.913036,0.019771,0.018355,0.090779,-0.002716,0.419670,-0.258335,0.102476,0.316994,...,0.003116,-0.072992,0.060094,-0.005568,-0.085019,0.356508,-0.155389,0.258998,0.444437,0
2,-0.317572,-0.751546,-0.018182,-0.012594,-0.113059,-0.011345,-0.353239,0.267224,-0.176562,-0.375043,...,0.003061,-0.137113,-0.099480,-0.150381,-0.110743,-0.686650,-0.121376,-0.693794,-0.674594,0
3,-0.410492,-0.353808,-0.010601,-0.008254,-0.077122,-0.006218,-0.201272,-0.216044,-0.219519,-0.172305,...,0.003110,-0.144686,-0.084676,-0.150402,-0.131186,-0.299561,-0.156966,-0.300753,-0.265660,1
4,-0.439214,-0.349547,-0.005928,-0.010028,-0.077591,-0.006223,-0.204675,-0.258335,-0.232585,-0.172305,...,0.003116,-0.144686,-0.084676,-0.150402,-0.131186,-0.299561,-0.156966,-0.300753,-0.265660,0


In [20]:
### Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device
RANDOMSEED = 42
random.seed(RANDOMSEED)
np.random.seed(RANDOMSEED)
torch.manual_seed(RANDOMSEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(RANDOMSEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
### Creating a generator to pass into the data loader 
g = torch.Generator()
g.manual_seed(RANDOMSEED)
batch_size = 64

In [21]:
X = global_dataset.drop(columns ='Label_Binary')
X = X.to_numpy()
y = global_dataset['Label_Binary']
y = y.to_numpy()
X_tensor = torch.tensor(X, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype = torch.long)
global_tuple = TensorDataset(X_tensor,y_tensor)
test_loader = DataLoader(global_tuple, batch_size=batch_size, shuffle=True)

### Loading all the models

In [22]:
centralized_model = utils.loading_pickle('./models/centralized_model.pickle')
fed_nova_model = utils.loading_pickle('./models/fednova.pickle')
fed_avg_model = utils.loading_pickle('./models/fedprox.pickle')
fed_prox_model = utils.loading_pickle('./models/fedavg.pickle')

In [23]:
### For model evaluation 
criterion = nn.CrossEntropyLoss()
def post_trained_global_model(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0.0
    total = 0 
    correct = 0 
    true_labels = []
    prediction = []

    with torch.no_grad():
        for samples, features in test_loader:
            samples = samples.to(device)
            features = features.to(device)
            output = model(samples)
            loss = criterion(output, features)
            _, predicted = output.max(1)
            prediction.extend(predicted.tolist())
            total += features.size(0)
            test_loss += loss.item()
            correct += predicted.eq(features).sum().item()
            true_labels.extend(features.tolist())

        test_loss = test_loss/len(test_loader.dataset)
        accuracy = 100* correct / total 
    
    return test_loss, accuracy, prediction, true_labels

In [27]:
### Centralized model 
centralized_test_loss, centralized_accuracy, centralized_prediction, centralized_true_labels = post_trained_global_model(centralized_model, test_loader, criterion, device)
print(centralized_test_loss)
print(centralized_accuracy)
print(classification_report(centralized_prediction, centralized_true_labels))

0.010210337947989276
87.73091022354266
              precision    recall  f1-score   support

           0       0.94      0.84      0.88     93115
           1       0.82      0.93      0.87     73833

    accuracy                           0.88    166948
   macro avg       0.88      0.88      0.88    166948
weighted avg       0.88      0.88      0.88    166948



In [31]:
fed_nova_test_loss, fed_nova_accuracy, fed_nova_prediction, fed_nova_true_labels = post_trained_global_model(fed_nova_model, test_loader, criterion, device)
print(fed_nova_test_loss)
print(fed_nova_accuracy)
print(classification_report(fed_nova_prediction, fed_nova_true_labels))

0.0055242360546754014
85.34693437477537
              precision    recall  f1-score   support

           0       0.97      0.79      0.87    102485
           1       0.74      0.96      0.83     64463

    accuracy                           0.85    166948
   macro avg       0.85      0.87      0.85    166948
weighted avg       0.88      0.85      0.86    166948



In [33]:
fed_prox_test_loss, fed_prox_accuracy, fed_prox_prediction, fed_prox_true_labels = post_trained_global_model(fed_prox_model, test_loader, criterion, device)
print(fed_prox_test_loss)
print(fed_prox_accuracy)
print(classification_report(fed_prox_prediction, fed_prox_true_labels))

0.0019374787805735258
96.48034118408127
              precision    recall  f1-score   support

           0       0.98      0.95      0.97     86044
           1       0.95      0.98      0.96     80904

    accuracy                           0.96    166948
   macro avg       0.96      0.97      0.96    166948
weighted avg       0.97      0.96      0.96    166948



In [35]:
fed_avg_test_loss, fed_avg_accuracy, fed_avg_prediction, fed_avg_true_labels = post_trained_global_model(fed_avg_model, test_loader, criterion, device)
print(fed_avg_test_loss)
print(fed_avg_accuracy)
print(classification_report(fed_avg_prediction, fed_avg_true_labels))

0.003479882171419713
92.00888899537581
              precision    recall  f1-score   support

           0       0.97      0.88      0.92     91953
           1       0.87      0.97      0.92     74995

    accuracy                           0.92    166948
   macro avg       0.92      0.92      0.92    166948
weighted avg       0.93      0.92      0.92    166948

